# Automatic Differentiation

Research on the topic of Automatic Differentiation and Backpropagation with `torch.autograd`.

In this Backpropagation, parameters (model weights) are adjusted according to the gradient of the loss function with respect to the given parameter.

`torch.autograd` supports automatic computation of gradient for any computational graph.


# Notebook Setup

## Imports

In [1]:
# Import Standard Libraries
import torch

# Tensors

## Basic Example

In [2]:
# Input and expected output
x = torch.ones(5)
y = torch.zeros(3)

# Parameters
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)

# Feed forward
z = torch.matmul(x, w)+b

# Compute loss
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

`grad_fn` is a reference to the Backpropagation function.

In [3]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

Gradient function for z = <AddBackward0 object at 0x107eff970>
Gradient function for loss = <BinaryCrossEntropyWithLogitsBackward0 object at 0x107eff940>


The function `backward()` of the loss function compute the gradients of the model's parameters.

In [4]:
# Compute the gradients of the model's parameters
loss.backward()
print(w.grad)
print(b.grad)

tensor([[0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215],
        [0.2574, 0.0718, 0.0215]])
tensor([0.2574, 0.0718, 0.0215])


The `requires_grad` is used to track the history of the computations and support gradients computation. Sometimes it's not required and it is possible to stop it.

In [5]:
# With computation tracking
z = torch.matmul(x, w)+b
print(z.requires_grad)

# Without computation tracking
with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

True
False


In [6]:
# Same effect but with 'detach' function
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

False


# Computational Graph

## Definition
Conceptually, autograd keeps a record of data (tensors) and all executed operations (along with the resulting new tensors) in a directed acyclic graph (DAG) consisting of Function objects. In this DAG, leaves are the input tensors, roots are the output tensors. By tracing this graph from roots to leaves, you can automatically compute the gradients using the chain rule.

## Feed Forward Step
Two operations at the same time:
- Compute the resulting tensor
- Store in the DAG the operation's gradient function (`grad_fn`)

## Backward
This operation is called over the root of the DAG. The `autograd` then does:
- Compute the gradient through `.grad_fn`
- Accumulate them in `.grad` attribute
- Using the chain rule, propagates all the way to the leaf tensors